# Capítulo 4 — Pandas II: limpieza, transformación, agrupación y combinación

**Serie Ciencia de Datos · Tomo 2 — Herramientas Python para Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.

## Código del capítulo

### Introducción del capítulo

In [ ]:
!pip install -q pandas==2.3.3

### 4.1 Valores faltantes: detección y criterio para decidir entre `dropna()` e `fillna()`

In [ ]:
import pandas as pd
import numpy as np

ventas = pd.DataFrame({
    "producto": ["Teclado", "Monitor", "Mouse", "Webcam", "Teclado"],
    "unidades": [12, np.nan, 30, 8, 12],
    "precio": [25.0, 180.0, np.nan, 45.0, 25.0],
    "region": ["Norte", "Sur", None, "Norte", "Norte"],
})

# Cuántos faltantes hay en CADA columna
print(ventas.isna().sum())
# Total en todo el DataFrame
print(ventas.isna().sum().sum())
# PORCENTAJE por columna, que es lo decisivo
print((ventas.isna().mean() * 100).round(1))

#### La estrategia de eliminación: `dropna()`

In [ ]:
# elimina toda fila con AL MENOS un faltante
print(ventas.dropna())
print(ventas.dropna(subset=["precio"]))     # solo mira la columna 'precio'
# elimina COLUMNAS con faltantes (axis, como en el Cap. 2)
print(ventas.dropna(axis=1))
# conserva filas con al menos 3 valores NO nulos
print(ventas.dropna(thresh=3))

#### La estrategia de imputación: `fillna()`

In [ ]:
relleno = ventas.copy()   # se imputa sobre una copia, no sobre el original

# Numérica: la MEDIANA suele ser preferible a la media, por ser robusta a
# atípicos
relleno["unidades"] = relleno["unidades"].fillna(relleno["unidades"].median())

# Categórica: una etiqueta explícita, que preserva la información de que el
# dato faltaba
relleno["region"] = relleno["region"].fillna("Desconocida")

In [ ]:
serie = pd.Series([10.0, np.nan, np.nan, 40.0])
print(serie.ffill().tolist())
# [10.0, 10.0, 10.0, 40.0]  -> arrastra el último válido
print(serie.interpolate().tolist())
# [10.0, 20.0, 30.0, 40.0]  -> interpola linealmente

### 4.2 Duplicados, tipos incorrectos y normalización de texto con el accesor `.str`

#### Duplicados

In [ ]:
import pandas as pd
import numpy as np

# La misma tabla de la Sección 4.1, con un identificador de pedido
# y una última fila que repite por completo la primera
ventas = pd.DataFrame({
    "id_pedido": [101, 102, 103, 104, 101],
    "producto": ["Teclado", "Monitor", "Mouse", "Webcam", "Teclado"],
    "unidades": [12, np.nan, 30, 8, 12],
    "precio": [25.0, 180.0, np.nan, 45.0, 25.0],
})

# Cuántas filas están repetidas por completo
print(ventas.duplicated().sum())
# Forma de la tabla tras eliminar las repeticiones exactas
print(ventas.drop_duplicates().shape)
# Repeticiones según una CLAVE de negocio
print(ventas.duplicated(subset=["producto"]).sum())
# Elimina las repeticiones de id_pedido y conserva la última aparición
print(ventas.drop_duplicates(subset=["id_pedido"], keep="last"))

#### Normalización de texto con `.str`

In [ ]:
sucio = pd.DataFrame({"region": ["  Norte ", "NORTE", "norte", "Sur",
                                 "sur "]})
print(sucio["region"].value_counts().to_dict())
# {'  Norte ': 1, 'NORTE': 1, 'norte': 1, 'Sur': 1, 'sur ': 1}
# -> 5 categorías... aparentes

sucio["region"] = sucio["region"].str.strip().str.lower().str.capitalize()
print(sucio["region"].value_counts().to_dict())
# {'Norte': 3, 'Sur': 2}   -> las categorías reales eran 2

In [ ]:
# Máscara booleana: ¿contiene el patrón?
print(sucio["region"].str.contains("or").tolist())
# [True, True, True, False, False]

# Longitud de cada cadena
print(sucio["region"].str.len().tolist())
# [5, 5, 5, 3, 3]

# Sustitución literal
print(pd.Series(["A-1", "B-2"]).str.replace("-", "_", regex=False).tolist())
# ['A_1', 'B_2']

# Dividir en varias columnas
print(pd.Series(["a,b", "c,d"]).str.split(",", expand=True))
#    0  1
# 0  a  b
# 1  c  d

#### Datos categóricos como decisión de limpieza y modelado

In [ ]:
tallas = pd.Series(["M", "S", "L", "M", "XL"])
orden = pd.CategoricalDtype(categories=["S", "M", "L", "XL"], ordered=True)
tallas_ordinales = tallas.astype(orden)

print((tallas_ordinales > "M").tolist())
# [False, False, True, False, True]
print(tallas_ordinales.sort_values().tolist())
# ['S', 'M', 'M', 'L', 'XL']  -> orden real, no alfabético
print(tallas_ordinales.min(), tallas_ordinales.max())   # S XL

In [ ]:
texto = pd.Series(["Norte"] * 50_000 + ["Sur"] * 50_000)
como_object = texto.memory_usage(deep=True)
como_category = texto.astype("category").memory_usage(deep=True)
print(f"como object  : {como_object / 1024**2:.2f} MB")
print(f"como category: {como_category / 1024**2:.3f} MB")
print(f"ahorro       : {(1 - como_category / como_object) * 100:.1f}%")

*Salida en el libro:*

```text
como object  : 5.05 MB
como category: 0.096 MB
ahorro       : 98.1%
```

### 4.3 Creación de variables: la escalera de decisión

#### Peldaño 1 — Operación vectorizada directa

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 200_000
df = pd.DataFrame({
    "precio": rng.random(n) * 100,
    "unidades": rng.integers(1, 50, n),
    "categoria": rng.choice(["A", "B", "C"], n),
})
df["costo"] = df["precio"] * df["unidades"] * rng.uniform(0.4, 0.9, n)

In [ ]:
# aritmética entre columnas
df["ingreso"] = df["precio"] * df["unidades"]
df["margen"] = (df["ingreso"] - df["costo"]) / df["ingreso"]
# comparación -> columna booleana
df["es_caro"] = df["precio"] > 100

#### Peldaño 2 — `.map()` y `.replace()` para sustituciones

In [ ]:
mapa_categorias = {"A": "Electrónica", "B": "Hogar", "C": "Ropa"}
df["nombre_categoria"] = df["categoria"].map(mapa_categorias)

#### Peldaño 3 — `np.where()` y `np.select()` para lógica condicional

In [ ]:
df["nivel"] = np.where(df["ingreso"] > 1500, "Alto", "Bajo")

In [ ]:
condiciones = [df["ingreso"] > 2000, df["ingreso"] > 1000,
               df["ingreso"] > 500]
opciones = ["Premium", "Alto", "Medio"]
df["segmento"] = np.select(condiciones, opciones, default="Bajo")

In [ ]:
datos = pd.DataFrame({"ingreso": [2500.0, 1500.0, 700.0, 100.0, np.nan]})
condiciones = [datos["ingreso"] > 2000, datos["ingreso"] > 1000,
               datos["ingreso"] > 500]
opciones = ["Premium", "Alto", "Medio"]
datos["segmento"] = np.select(condiciones, opciones, default="Bajo")
print(datos)
#    ingreso segmento
# 0   2500.0  Premium
# 1   1500.0     Alto
# 2    700.0    Medio
# 3    100.0     Bajo
# 4      NaN     Bajo   -> un dato AUSENTE, igual que la fila 3

In [ ]:
condiciones = [datos["ingreso"].isna(), datos["ingreso"] > 2000,
               datos["ingreso"] > 1000, datos["ingreso"] > 500]
opciones = ["Sin dato", "Premium", "Alto", "Medio"]
datos["segmento"] = np.select(condiciones, opciones, default="Bajo")
print(datos)
#    ingreso  segmento
# 0   2500.0   Premium
# 1   1500.0      Alto
# 2    700.0     Medio
# 3    100.0      Bajo
# 4      NaN  Sin dato   -> ahora el faltante conserva su identidad

#### Peldaño 4 — `apply()`, solo cuando no hay equivalente vectorizado

In [ ]:
import timeit

def con_apply_fila(datos):
    return datos.apply(lambda fila: fila["precio"] * fila["unidades"], axis=1)

def con_itertuples(datos):
    return [fila.precio * fila.unidades for fila in datos.itertuples()]

def vectorizado(datos):
    return datos["precio"] * datos["unidades"]

tiempo_apply_fila = min(timeit.repeat(
    lambda: con_apply_fila(df), repeat=3, number=1))
tiempo_itertuples = min(timeit.repeat(
    lambda: con_itertuples(df), repeat=3, number=1))
tiempo_vectorizado = min(timeit.repeat(
    lambda: vectorizado(df), repeat=7, number=10)) / 10

print(f"apply(axis=1)      : {tiempo_apply_fila * 1000:9.1f} ms")
print(f"bucle itertuples() : {tiempo_itertuples * 1000:9.1f} ms")
print(f"vectorizado        : {tiempo_vectorizado * 1000:9.3f} ms")
print(f"apply(axis=1) frente a vectorizado : "
      f"{tiempo_apply_fila / tiempo_vectorizado:.0f} veces más lento")
print(f"itertuples() frente a vectorizado  : "
      f"{tiempo_itertuples / tiempo_vectorizado:.0f} veces más lento")
print(f"apply(axis=1) frente a itertuples(): "
      f"{tiempo_apply_fila / tiempo_itertuples:.0f} veces más lento")

*Salida en el libro:*

```text
apply(axis=1)      :     891.2 ms
bucle itertuples() :     141.6 ms
vectorizado        :     0.342 ms
apply(axis=1) frente a vectorizado : 2602 veces más lento
itertuples() frente a vectorizado  : 413 veces más lento
apply(axis=1) frente a itertuples(): 6 veces más lento
```

### 4.4 Ordenamiento y ranking: `sort_values()`, `rank()`, `nlargest()`

In [ ]:
import pandas as pd

datos = pd.DataFrame({
    "producto": ["Teclado", "Monitor", "Mouse", "Webcam", "Auriculares",
                 "Hub"],
    "region": ["Norte", "Sur", "Norte", "Sur", "Este", "Este"],
    "ingreso": [300.0, 900.0, 375.0, 360.0, 520.0, 150.0],
    "margen": [0.35, 0.22, 0.41, 0.18, 0.30, 0.12],
})

In [ ]:
# una columna, descendente
print(datos.sort_values("ingreso", ascending=False))
# jerárquico
print(datos.sort_values(["region", "ingreso"], ascending=[True, False]))

In [ ]:
print(pd.Series([10, 20, 20, 30]).rank(method="dense").tolist())
# [1.0, 2.0, 2.0, 3.0]
print(pd.Series([10, 20, 20, 30]).rank(method="min").tolist())
# [1.0, 2.0, 2.0, 4.0]

In [ ]:
print(datos.nlargest(5, "ingreso"))       # top 5 por ingreso
print(datos.nsmallest(3, "margen"))       # los 3 peores por margen

### 4.5 `groupby()` y el patrón split–apply–combine

In [ ]:
import pandas as pd

transacciones = pd.DataFrame({
    "region": ["Norte", "Sur", "Norte", "Sur", "Este", "Norte"],
    "categoria": ["A", "A", "B", "B", "A", "A"],
    "ingreso": [100.0, 200.0, 150.0, 300.0, 50.0, 120.0],
    "unidades": [1, 2, 3, 4, 1, 2],
})

In [ ]:
print(transacciones.groupby("region")["ingreso"].sum())
# region
# Este      50.0
# Norte    370.0
# Sur      500.0
# Name: ingreso, dtype: float64

In [ ]:
print(transacciones.groupby(["region", "categoria"])["ingreso"].sum())

In [ ]:
# resultado como DataFrame plano
transacciones.groupby("region", as_index=False)["ingreso"].sum()
# tamaño de cada grupo
transacciones.groupby("region").size()
# inspeccionar UN grupo concreto
transacciones.groupby("region").get_group("Norte")

In [ ]:
transacciones["media_region"] = (transacciones.groupby("region")["ingreso"]
                                 .transform("mean"))
transacciones["desviacion"] = (transacciones["ingreso"]
                               - transacciones["media_region"])

### 4.6 Agregaciones múltiples con `agg()` y tablas dinámicas con `pivot_table()`

#### `agg()`: varias métricas de una vez

In [ ]:
import pandas as pd

# Las mismas seis transacciones de la Sección 4.5
transacciones = pd.DataFrame({
    "region": ["Norte", "Sur", "Norte", "Sur", "Este", "Norte"],
    "categoria": ["A", "A", "B", "B", "A", "A"],
    "ingreso": [100.0, 200.0, 150.0, 300.0, 50.0, 120.0],
    "unidades": [1, 2, 3, 4, 1, 2],
})

print(transacciones.groupby("region")["ingreso"]
      .agg(["sum", "mean", "count"]))

In [ ]:
resumen = transacciones.groupby("region").agg(
    ingreso_total=("ingreso", "sum"),
    ticket_promedio=("ingreso", "mean"),
    unidades_totales=("unidades", "sum"),
    numero_transacciones=("ingreso", "size"),
).round(2)
print(resumen)

*Salida en el libro:*

```text
        ingreso_total  ticket_promedio  unidades_totales  numero_transacciones
region
Este             50.0            50.00                 1                     1
Norte           370.0           123.33                 6                     3
Sur             500.0           250.00                 6                     2
```

#### `pivot_table()`: reorganizar en dos dimensiones

In [ ]:
print(pd.pivot_table(transacciones, values="ingreso", index="region",
                     columns="categoria", aggfunc="sum", fill_value=0,
                     margins=True))
# categoria      A      B    All
# region
# Este        50.0    0.0   50.0
# Norte      220.0  150.0  370.0
# Sur        200.0  300.0  500.0
# All        470.0  450.0  920.0

### 4.7 Combinación de tablas: `merge()` y los cuatro tipos de join, frente a `concat()`

#### `merge()` y los cuatro tipos de unión

In [ ]:
import pandas as pd

pedidos  = pd.DataFrame({"id_pedido": [1, 2, 3, 4],
                         "id_cliente": [10, 11, 10, 99],
                         "monto": [100, 200, 150, 80]})
clientes = pd.DataFrame({"id_cliente": [10, 11, 12],
                         "nombre": ["Ana", "Luis", "Eva"]})

In [ ]:
print(pd.merge(pedidos, clientes, on="id_cliente", how="left"))
#    id_pedido  id_cliente  monto nombre
# 0          1          10    100    Ana
# 1          2          11    200   Luis
# 2          3          10    150    Ana
# 3          4          99     80    NaN

In [ ]:
# indicator=True añade una columna que dice de dónde vino cada fila
auditoria = pd.merge(pedidos, clientes, on="id_cliente", how="outer",
                     indicator=True)
print(auditoria["_merge"].value_counts().to_dict())
# {'both': 3, 'left_only': 1, 'right_only': 1}
# -> cuantifica exactamente qué no casó

# validate declara la cardinalidad esperada y lanza error si no se cumple
unidas = pd.merge(pedidos, clientes, on="id_cliente", how="left",
                  validate="many_to_one")
print(unidas.shape)   # (4, 4)  -> se cumple: las mismas 4 filas de pedidos

In [ ]:
# Una tabla de clientes con el id 10 repetido
clientes_duplicados = pd.DataFrame({"id_cliente": [10, 10, 11],
                                    "nombre": ["Ana", "Ana B.", "Luis"]})
try:
    pd.merge(pedidos, clientes_duplicados, on="id_cliente", how="left",
             validate="many_to_one")
except pd.errors.MergeError as error:
    print(f"MergeError: {error}")

*Salida en el libro:*

```text
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
```

#### `concat()`: apilar, no cruzar

In [ ]:
enero = pd.DataFrame({"producto": ["Teclado"], "monto": [100]})
febrero = pd.DataFrame({"producto": ["Mouse"], "monto": [80]})
marzo = pd.DataFrame({"producto": ["Monitor"], "monto": [180]})

# Apila filas (axis=0, por defecto)
print(pd.concat([enero, febrero, marzo], ignore_index=True))
#   producto  monto
# 0  Teclado    100
# 1    Mouse     80
# 2  Monitor    180

# Yuxtapone columnas
tabla_a = pd.DataFrame({"x": [1, 2]})
tabla_b = pd.DataFrame({"y": [3, 4]})
print(pd.concat([tabla_a, tabla_b], axis=1))
#    x  y
# 0  1  3
# 1  2  4

### 4.8 Fechas y tiempo: `to_datetime()`, el accesor `.dt` y `resample()`

In [ ]:
import pandas as pd

datos = pd.DataFrame({
    "fecha": pd.to_datetime(["2024-01-15", "2024-01-20", "2024-02-03",
                             "2024-02-28", "2024-03-10"]),
    "ingreso": [100.0, 150.0, 200.0, 120.0, 300.0],
})

datos["anio"] = datos["fecha"].dt.year
datos["mes"] = datos["fecha"].dt.month
datos["trimestre"] = datos["fecha"].dt.quarter
datos["dia_semana"] = datos["fecha"].dt.day_name()
# 'Monday', 'Saturday', ...
datos["nombre_mes"] = datos["fecha"].dt.month_name()

In [ ]:
print((datos["fecha"].max() - datos["fecha"].min()).days)
# 55  -> rango temporal en días

In [ ]:
indexado = datos.set_index("fecha")
print(indexado["ingreso"].resample("ME").sum())    # total por MES (Month End)
# fecha
# 2024-01-31    250.0
# 2024-02-29    320.0
# 2024-03-31    300.0
# Freq: ME, Name: ingreso, dtype: float64

In [ ]:
serie = datos["fecha"]
print(serie.dt.to_period("M").iloc[0])
# 2024-01  -> correcto, sin advertencia
try:
    serie.dt.to_period("ME")
except ValueError as error:
    print(f"ValueError: {error}")

# correcto, sin advertencia
por_mes = indexado["ingreso"].resample("ME").sum()
por_mes = indexado["ingreso"].resample("M").sum()    # emite una advertencia

*Salida en el libro:*

```text
2024-01
ValueError: Invalid frequency: ME, failed to parse with error message: ValueError("for Period, please use 'M' instead of 'ME'")
FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  por_mes = indexado["ingreso"].resample("M").sum()    # emite una advertencia
```

In [ ]:
print(indexado.loc["2024-02"])          # todas las filas de febrero de 2024
print(indexado.loc["2024-02":"2024-03"])   # rango de meses

## Soluciones de los ejercicios

### [Ex-L2-C4-01] Pipeline de limpieza completo

#### Desarrollo con código

In [ ]:
import pandas as pd
import numpy as np

generador = np.random.default_rng(2024)
FILAS = 3000

ventas_sucias = pd.DataFrame({
    "id_venta": np.arange(1, FILAS + 1),
    "fecha": pd.to_datetime("2024-01-01")
             + pd.to_timedelta(generador.integers(0, 365, FILAS), unit="D"),
    "vendedor": generador.choice(["  Ana Ruiz", "ANA RUIZ", "ana ruiz ",
                                  "Luis Paz", "LUIS PAZ", "Eva Soto",
                                  "eva soto"], FILAS),
    "categoria": generador.choice(["Electrónica", "Hogar", "Ropa"], FILAS),
    "unidades": generador.integers(1, 30, FILAS),
    # precio como TEXTO con coma decimal: tipo incorrecto deliberado
    "precio": [f"{v:.2f}".replace(".", ",")
               for v in (generador.random(FILAS) * 300 + 10)],
    "descuento": generador.random(FILAS).round(3),
})
# Faltantes en dos columnas
ventas_sucias.loc[generador.choice(FILAS, 180, replace=False),
                  "precio"] = None
ventas_sucias.loc[generador.choice(FILAS, 420, replace=False),
                  "descuento"] = np.nan
# Duplicados exactos
ventas_sucias = pd.concat([ventas_sucias,
                           ventas_sucias.sample(75, random_state=1)],
                          ignore_index=True)
# Duplicados por clave de negocio: mismo id_venta, distintas unidades
conflicto = ventas_sucias.sample(20, random_state=2).copy()
conflicto["unidades"] = conflicto["unidades"] + 5
ventas_sucias = pd.concat([ventas_sucias, conflicto], ignore_index=True)
ventas_sucias = (ventas_sucias.sample(frac=1, random_state=3)
                 .reset_index(drop=True))

# ---------------------------------------------
# FASE 1 — DIAGNÓSTICO (medir antes de tocar nada)
# ---------------------------------------------
print(ventas_sucias.shape)                              # (3095, 7)
# precio es object: sospechoso
print(ventas_sucias.dtypes)
# precio 6.01%, descuento 13.96%
print((ventas_sucias.isna().mean() * 100).round(2))
print(ventas_sucias.duplicated().sum())
# 75 duplicados EXACTOS
print(ventas_sucias.duplicated(subset=["id_venta"]).sum())
# 95 por CLAVE de negocio
print(sorted(ventas_sucias["vendedor"].unique()))
# Son 7 cadenas distintas... para 3 personas reales.

*Salida en el libro:*

```text
(3095, 7)
id_venta              int64
fecha        datetime64[ns]
vendedor             object
categoria            object
unidades              int64
precio               object
descuento           float64
dtype: object
id_venta      0.00
fecha         0.00
vendedor      0.00
categoria     0.00
unidades      0.00
precio        6.01
descuento    13.96
dtype: float64
75
95
['  Ana Ruiz', 'ANA RUIZ', 'Eva Soto', 'LUIS PAZ', 'Luis Paz', 'ana ruiz ', 'eva soto']
```

In [ ]:
# ---------------------------------------------
# FASE 2 — LIMPIEZA, CON REGISTRO DE CADA DECISIÓN
# ---------------------------------------------
limpio = ventas_sucias.copy()
registro = []

# DECISIÓN 1 — Duplicados exactos.
# Filas idénticas en las 7 columnas: no puede ser una venta legítima repetida,
# porque el id_venta también coincide. Es un error de carga. Se eliminan.
antes = len(limpio)
limpio = limpio.drop_duplicates()
registro.append(("Duplicados exactos eliminados", antes - len(limpio)))

# DECISIÓN 2 — Normalización de la columna 'vendedor'.
# strip() quita los espacios sobrantes (invisibles y por eso traicioneros)
# y title() unifica la capitalización. 7 variantes -> 3 personas.
variantes = limpio["vendedor"].nunique()
limpio["vendedor"] = limpio["vendedor"].str.strip().str.title()
registro.append((f"Vendedores unificados ({variantes} variantes -> "
                 f"{limpio['vendedor'].nunique()} personas)",
                 variantes - limpio["vendedor"].nunique()))

# DECISIÓN 3 — Duplicados por CLAVE DE NEGOCIO (distinto de los exactos).
# Mismo id_venta con 'unidades' diferente: NO es una repetición, es una
# INCONSISTENCIA. Sin acceso al sistema de origen no se puede saber cuál
# registro es el correcto. Se conserva el primero y se DOCUMENTA la
# ambigüedad.
conflictivos = limpio.duplicated(subset=["id_venta"], keep=False).sum()
antes = len(limpio)
limpio = limpio.drop_duplicates(subset=["id_venta"], keep="first")
registro.append((f"Conflictos por id_venta ({conflictivos} filas implicadas)",
                 antes - len(limpio)))

# DECISIÓN 4 — Tipo incorrecto: 'precio' llegó como texto con coma decimal.
# errors="coerce" convierte a NaN lo que no se pueda parsear, en lugar de
# lanzar una excepción: aquí interesa continuar y contabilizar los fallos.
limpio["precio"] = pd.to_numeric(
    limpio["precio"].str.replace(",", ".", regex=False), errors="coerce")
registro.append(("'precio' convertido a numérico; quedaron como NaN",
                 int(limpio["precio"].isna().sum())))

# DECISIÓN 5 — Faltantes en 'precio': ELIMINAR.
# Es la variable central del análisis. Imputarla con la media significaría
# inventar precisamente el dato que se quiere medir.
antes = len(limpio)
limpio = limpio.dropna(subset=["precio"])
registro.append(("Filas sin precio eliminadas", antes - len(limpio)))

# DECISIÓN 6 — Faltantes en 'descuento': IMPUTAR con 0.0.
# Aquí el faltante es INFORMATIVO: el sistema deja el campo vacío cuando
# no se aplicó descuento. Un 0.0 no inventa nada, recupera el significado.
faltan_descuento = int(limpio["descuento"].isna().sum())
limpio["descuento"] = limpio["descuento"].fillna(0.0)
registro.append(("Descuento imputado con 0.0 (ausencia = sin descuento)",
                 faltan_descuento))

print("Registro de limpieza:")
for numero, (decision, afectadas) in enumerate(registro, start=1):
    print(f"  {numero}. {decision}: {afectadas}")
print("\nBalance final:")
print(f"  filas     : {len(ventas_sucias)} -> {len(limpio)}"
      f"   (retenidas {len(limpio) / len(ventas_sucias):.1%})")
print("  vendedores:", sorted(limpio["vendedor"].unique()))
print("  precio    :", ventas_sucias["precio"].dtype, "->",
      limpio["precio"].dtype)
print("  faltantes :", limpio.isna().sum().sum())

#### Salida esperada

*Salida en el libro:*

```text
Registro de limpieza:
  1. Duplicados exactos eliminados: 75
  2. Vendedores unificados (7 variantes -> 3 personas): 4
  3. Conflictos por id_venta (40 filas implicadas): 20
  4. 'precio' convertido a numérico; quedaron como NaN: 180
  5. Filas sin precio eliminadas: 180
  6. Descuento imputado con 0.0 (ausencia = sin descuento): 397

Balance final:
  filas     : 3095 -> 2820   (retenidas 91.1%)
  vendedores: ['Ana Ruiz', 'Eva Soto', 'Luis Paz']
  precio    : object -> float64
  faltantes : 0
```

### [Ex-L2-C4-02] Análisis con `groupby()`

#### Desarrollo con código

In [ ]:
import pandas as pd

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
limpio = pd.read_csv(DATOS + "ventas_limpias_c4.csv", parse_dates=["fecha"])

# ---------------------------------------------
# 1. INGRESO POR CATEGORÍA Y REGIÓN (agrupación por DOS claves)
# ---------------------------------------------
# observed=True evita que las categóricas generen combinaciones vacías
por_categoria_region = limpio.groupby(["categoria", "region"],
                                      observed=True).agg(
    # nombre_resultado=("columna", "función")
    ingreso_total=("ingreso", "sum"),
    ticket_promedio=("ingreso", "mean"),
    ventas=("id_venta", "size"),
).round(2)

# ---------------------------------------------
# 2. TICKET PROMEDIO POR REGIÓN
# ---------------------------------------------
ticket_region = limpio.groupby("region", observed=True).agg(
    ticket_promedio=("ingreso", "mean"),
    ingreso_total=("ingreso", "sum"),
    ventas=("id_venta", "size"),
).round(2).sort_values("ticket_promedio", ascending=False)

# ---------------------------------------------
# 3. TOP 5 PRODUCTOS POR INGRESO ACUMULADO
# ---------------------------------------------
# nlargest() es más directo y eficiente que ordenar la tabla completa
top_productos = (limpio.groupby("producto", observed=True)["ingreso"]
                 .sum().round(2).nlargest(5))

# ---------------------------------------------
# 4. TABLA DINÁMICA CON TOTALES MARGINALES
# ---------------------------------------------
tabla = pd.pivot_table(limpio, values="ingreso", index="region",
                       columns="categoria", aggfunc="sum", margins=True,
                       margins_name="TOTAL", observed=True).round(0)

# ---------------------------------------------
# 5. TRANSFORM: comparar cada venta con su propio grupo
# ---------------------------------------------
# transform() devuelve un resultado de la MISMA forma que el original,
# propagando el valor del grupo a cada una de sus filas (Sección 4.5).
limpio["ticket_medio_region"] = limpio.groupby(
    "region", observed=True)["ingreso"].transform("mean")
limpio["desviacion"] = limpio["ingreso"] - limpio["ticket_medio_region"]

# ---------------------------------------------
# 6. SERIE MENSUAL
# ---------------------------------------------
# Nótese "M" y no "ME": to_period usa el alias de PERIODO (Sección 4.8)
limpio["mes"] = limpio["fecha"].dt.to_period("M")
serie_mensual = limpio.groupby("mes", observed=True)["ingreso"].sum().round(0)

print("Ticket promedio por región:")
print(ticket_region)
print("\nTop 5 productos por ingreso acumulado:")
print(top_productos)
print("\nTabla dinámica con totales marginales:")
print(tabla)
print("\nVerificación de transform(): suma de desviaciones por región")
print(limpio.groupby("region", observed=True)["desviacion"].sum().round(6))

#### Salida esperada

*Salida en el libro:*

```text
Ticket promedio por región:
        ticket_promedio  ingreso_total  ventas
region                                        
Norte           1378.70     1313898.83     953
Sur             1354.22     1278387.26     944
Este            1339.04     1235933.25     923

Top 5 productos por ingreso acumulado:
producto
Monitor        603525.10
Auriculares    573745.76
Tablet         563517.83
Impresora      548572.58
Webcam         527526.63
Name: ingreso, dtype: float64

Tabla dinámica con totales marginales:
categoria  Electrónica      Hogar       Ropa      TOTAL
region                                                 
Este          437976.0   427012.0   370946.0  1235933.0
Norte         450445.0   421626.0   441828.0  1313899.0
Sur           402015.0   431747.0   444626.0  1278387.0
TOTAL        1290435.0  1280385.0  1257400.0  3828219.0

Verificación de transform(): suma de desviaciones por región
region
Este     0.0
Norte    0.0
Sur     -0.0
Name: desviacion, dtype: float64
```

### [Ex-L2-C4-03] Integración de tres tablas

#### Desarrollo con código

In [ ]:
import pandas as pd

pedidos = pd.DataFrame({
    "id_pedido": [1, 2, 3, 4, 5, 6],
    # el 99 NO existe en clientes
    "id_cliente": [10, 11, 10, 99, 12, 11],
    # P9 NO existe en productos
    "id_producto": ["P1", "P2", "P1", "P3", "P9", "P2"],
    "unidades": [2, 1, 5, 3, 1, 4],
})
clientes = pd.DataFrame({
    "id_cliente": [10, 11, 12, 13],                  # el 13 nunca compró
    "nombre": ["Ana", "Luis", "Eva", "Marta"],
    "segmento": ["Premium", "Estándar", "Premium", "Estándar"],
})
productos = pd.DataFrame({
    "id_producto": ["P1", "P2", "P3", "P4"],         # P4 nunca se pidió
    "descripcion": ["Teclado", "Monitor", "Mouse", "Webcam"],
    "precio": [25.0, 180.0, 12.5, 45.0],
})

# ---------------------------------------------
# UNIÓN 1 — pedidos + clientes
# ---------------------------------------------
# PASO PREVIO: auditar con outer + indicator para saber QUÉ no casa
auditoria = pd.merge(pedidos, clientes, on="id_cliente", how="outer",
                     indicator=True)
print(auditoria["_merge"].value_counts().to_dict())
# {'both': 5, 'left_only': 1, 'right_only': 1}
#   left_only  = 1 pedido con un cliente inexistente (el 99)
#   right_only = 1 cliente que nunca compró (el 13)

# DECISIÓN: how="left".
# La tabla de pedidos es la tabla de HECHOS: cada fila es una transacción real
# que ocurrió y debe contabilizarse, aunque su cliente no esté en el maestro.
# Perder el pedido 4 falsearía el ingreso total. El cliente 13 sí puede quedar
# fuera: no aporta nada a un análisis de ventas.
paso_1 = pd.merge(pedidos, clientes, on="id_cliente", how="left",
                  validate="many_to_one")   # muchos pedidos -> un cliente

print(pedidos.shape, "->", paso_1.shape)    # (6, 4) -> (6, 6): filas intactas

# ---------------------------------------------
# UNIÓN 2 — resultado + productos (mismo razonamiento)
# ---------------------------------------------
completo = pd.merge(paso_1, productos, on="id_producto", how="left",
                    validate="many_to_one")
print(paso_1.shape, "->", completo.shape)   # (6, 6) -> (6, 8): filas intactas
completo["importe"] = completo["precio"] * completo["unidades"]
print(completo)

#### Salida esperada

*Salida en el libro:*

```text
{'both': 5, 'left_only': 1, 'right_only': 1}
(6, 4) -> (6, 6)
(6, 6) -> (6, 8)
   id_pedido  id_cliente id_producto  unidades nombre  segmento descripcion  \
0          1          10          P1         2    Ana   Premium     Teclado   
1          2          11          P2         1   Luis  Estándar     Monitor   
2          3          10          P1         5    Ana   Premium     Teclado   
3          4          99          P3         3    NaN       NaN       Mouse   
4          5          12          P9         1    Eva   Premium         NaN   
5          6          11          P2         4   Luis  Estándar     Monitor   

   precio  importe  
0    25.0     50.0  
1   180.0    180.0  
2    25.0    125.0  
3    12.5     37.5  
4     NaN      NaN  
5   180.0    720.0  
```

In [ ]:
# Si la tabla de clientes tuviera un id_cliente duplicado (error de datos
# frecuente):
clientes_duplicado = pd.concat([clientes, pd.DataFrame({
    "id_cliente": [10], "nombre": ["Ana (dup)"], "segmento": ["Premium"]})])

sin_validar = pd.merge(pedidos, clientes_duplicado, on="id_cliente",
                       how="left")
print(len(pedidos), "->", len(sin_validar))   # 6 -> 8   ¡dos filas fantasma!

total_correcto = pedidos["unidades"].sum()
total_inflado = sin_validar["unidades"].sum()
print(f"unidades: {total_correcto} -> {total_inflado}"
      f" (+{total_inflado / total_correcto - 1:.1%})")

# Con validate, el error es inmediato y explícito:
try:
    pd.merge(pedidos, clientes_duplicado, on="id_cliente", how="left",
             validate="many_to_one")
except pd.errors.MergeError as error:
    print(f"MergeError: {error}")

*Salida en el libro:*

```text
6 -> 8
unidades: 16 -> 23 (+43.8%)
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
```

### [Ex-L2-C4-04] Benchmark de la escalera de decisión

#### Desarrollo con código

In [ ]:
import pandas as pd
import numpy as np
import timeit

generador = np.random.default_rng(42)
N = 200_000
datos = pd.DataFrame({
    "categoria": generador.choice(["A", "B", "C", "D"], N),
    "importe": generador.random(N) * 1000,
})
MAPA_IVA = {"A": 0.21, "B": 0.10, "C": 0.04, "D": 0.00}


# PELDAÑO 2 — .map() con diccionario, luego aritmética vectorizada
def via_map(df):
    return df["importe"] * (1 + df["categoria"].map(MAPA_IVA))


# PELDAÑO 3 — np.select con una condición por categoría
def via_select(df):
    condiciones = [df["categoria"] == c for c in ["A", "B", "C", "D"]]
    opciones = [0.21, 0.10, 0.04, 0.00]
    return df["importe"] * (1 + np.select(condiciones, opciones, default=0.0))


# PELDAÑO 4 — apply(axis=1): construye un objeto Series por CADA fila
def via_apply_axis1(df):
    return df.apply(
        lambda fila: fila["importe"] * (1 + MAPA_IVA[fila["categoria"]]),
        axis=1)


# BUCLE EXPLÍCITO — la referencia contra la que comparar apply(axis=1)
def via_itertuples(df):
    return [fila.importe * (1 + MAPA_IVA[fila.categoria])
            for fila in df.itertuples()]


# ---------------------------------------------
# VERIFICAR EQUIVALENCIA ANTES DE MEDIR
# ---------------------------------------------
referencia = via_map(datos).values
for nombre, funcion in [("np.select", via_select),
                        ("apply(axis=1)", via_apply_axis1),
                        ("itertuples", via_itertuples)]:
    print(f"{nombre:<14}",
          np.allclose(referencia, np.asarray(funcion(datos))))


# ---------------------------------------------
# MEDIR (mínimo de repeticiones, Sección 1.7)
# ---------------------------------------------
def medir(funcion, repeticiones=5, ejecuciones=1):
    return min(timeit.repeat(funcion, repeat=repeticiones,
                             number=ejecuciones)) / ejecuciones

tiempos = {
    "Vectorizada + map": medir(lambda: via_map(datos), 5, 5),
    "np.select": medir(lambda: via_select(datos), 5, 5),
    "itertuples": medir(lambda: via_itertuples(datos), 5, 1),
    "apply(axis=1)": medir(lambda: via_apply_axis1(datos), 3, 1),
}
PELDANOS = {"Vectorizada + map": "2", "np.select": "3", "itertuples": "—",
            "apply(axis=1)": "4"}
mas_rapido = min(tiempos.values())
print(f"\n{'Enfoque':<20}{'Peldaño':>8}{'Tiempo (ms)':>13}{'Relativo':>10}")
for nombre, tiempo in sorted(tiempos.items(), key=lambda x: x[1]):
    print(f"{nombre:<20}{PELDANOS[nombre]:>8}{tiempo * 1000:>13.2f}"
          f"{tiempo / mas_rapido:>9.0f}x")

#### Salida esperada

*Salida en el libro:*

```text
np.select      True
apply(axis=1)  True
itertuples     True

Enfoque              Peldaño  Tiempo (ms)  Relativo
Vectorizada + map          2         8.42        1x
np.select                  3        55.46        7x
itertuples                 —       117.02       14x
apply(axis=1)              4      1243.66      148x
```